In [9]:
import torch
import torch.nn as nn
from einops import einsum

# Problem 5: 实现 softmax (1 point)
def softmax(x: torch.Tensor, i: int) -> torch.Tensor:
    '''
    x: 输入张量，支持任意维度
    i: int  待计算 softmax 的维度编号
    返回形状: 与输入 x 形状完全一致
    '''

    # .max() 的返回值是一个元组 (最大值张量, 最大值下标张量)，元组里的元素不可修改，列表里的元素可以修改
    x -= x.max(dim=i, keepdim=True)[0] # 假设 i = -1: x: (... dim) x.max: (... 1) -> (... dim)

    return torch.exp(x) / torch.exp(x).sum(dim=i,  keepdim=True) # 假设 i = -1: x: (... dim) x.sum: (... 1) -> (... dim)

In [10]:
# 测试 softmax
x_sm = torch.tensor([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]])
out_sm = softmax(x_sm, i=-1)
print("输入：\n", x_sm)
print("softmax输出：\n", out_sm)
print("每行求和：", out_sm.sum(dim=-1), "\n")

输入：
 tensor([[-2., -1.,  0.],
        [ 0.,  0.,  0.]])
softmax输出：
 tensor([[0.0900, 0.2447, 0.6652],
        [0.3333, 0.3333, 0.3333]])
每行求和： tensor([1., 1.]) 



In [11]:
# Problem 6: 实现缩放点积注意力 (5 points)
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor = None
) -> torch.Tensor:
    """
    实现缩放点积注意力
    q, k: (batch_size, ..., seq_len, d_k)
    v: (batch_size, ..., seq_len, d_v)
    mask: 可选布尔张量 (seq_len, seq_len)
          mask=True 代表该位置允许参与注意力权重计算, False 权重置零
    return: (batch_size, ..., seq_len, d_v)
    """

    # scores = q @ k.T
    scores = einsum(q, k, "... seq_len1 d, ... seq_len2 d -> ... seq_len1 seq_len2") # (..., seq_len1, d_k), (..., seq_len2, d_k) → (..., seq_len1, seq_len2)
   
    scores /= (q.shape[-1] ** 0.5)  # 使用 sqrt(d_k) 缩放，防止维度变大后内积过大导致 softmax 饱和
    
    if mask is not None:
        scores = scores.masked_fill(mask == False, -torch.inf) # 对 False 位置（对应于上三角除对角线上的元素）进行填充，填充值为负无穷，经过 softmax 后权重 ≈0
    
    scores = softmax(scores, -1) # 在最后一个维度上做 softmax

    # y = scores @ v
    y = einsum(scores, v, "... seq_len1 seq_len2, ... seq_len2 d -> ... seq_len1 d") # (..., seq_len1 seq_len2), (..., seq_len2 d_v) → (..., seq_len1 d_v)
    return y

In [12]:
# 测试 scaled_dot_product_attention 带因果掩码
B, S, dk, dv = 2, 4, 8, 8
q = torch.randn(B, S, dk)
k = torch.randn(B, S, dk)
v = torch.randn(B, S, dv)
# 因果下三角掩码
causal_mask = torch.tril(torch.ones(S, S)).bool()
attn_out = scaled_dot_product_attention(q, k, v, mask=causal_mask)
print(f"输入 q/k/v 形状: {q.shape}")
print(f"注意力输出 形状: {attn_out.shape}")

输入 q/k/v 形状: torch.Size([2, 4, 8])
注意力输出 形状: torch.Size([2, 4, 8])


In [13]:
import torch
import torch.nn as nn
from einops import rearrange
from transformer import Linear, RotaryPositionalEmbedding

# Problem 7: 实现因果多头自注意力 MHA (5 points)
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, max_seq_len: int, rope_theta: float) -> None:
        super().__init__()
        '''
        d_model: int  Transformer 块输入维度
        num_heads: int  注意力头数
        max_seq_len: int  RoPE 最大序列长度
        rope_theta: float  RoPE 基数频率
        '''

        self.d_model = d_model
        self.d_head = d_model // num_heads   # 计算每个头的向量维度
        self.num_heads = num_heads

        self.w_qkv = Linear(d_model, 3 * d_model) # 所有头的 qkv 矩阵合并成一个大矩阵: 输出维度 out_dim = num_heads * d_head * 3

        # rope_theta>0 启用旋转位置编码；d_head为每个头维度
        self.rope = RotaryPositionalEmbedding(theta=rope_theta, d_k=self.d_head, max_seq_len=max_seq_len) if rope_theta > 0 else None

        self.w_output = Linear(d_model, d_model) # 输出投影矩阵

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor=None) -> torch.Tensor:
        '''
        :param x: 输入张量: (batch_size, seq_len, d_model)
        :param token_positions: 每个 token 的位置下标 (batch_size, seq_len)，为 None 自动生成 0~seq_len‑1
        :return: (batch_size, seq_len, d_model)
        '''

        qkv = self.w_qkv(x) # (B, S, d_model) -> (B, S, 3*d_model)
        
        q, k, v = torch.split(qkv, self.d_model, dim=-1) # (B, S, 3*d_model) -> q/k/v: (B, S, d_model)

        # 拆分多头：把 d_model 拆成 num_heads × d_head，num_heads 放到 seq 前面，匹配 rope 的代码（回顾一下 rope 对输入 x 的要求）
        # (B, S, num_heads*d_head) → (B, num_heads, S, d_head)
        q = rearrange(q, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)
        k = rearrange(k, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)
        v = rearrange(v, "... seq (num_heads d_head) -> ... num_heads seq d_head", num_heads=self.num_heads, d_head=self.d_head)

        seq_len = x.shape[1]

        
        if self.rope is not None: # 是否使用 rope
            if token_positions is None:
                token_positions = torch.arange(seq_len, device=x.device) # 未传入位置时默认连续位置 0,1,2...seq_len‑1
            q = self.rope(q, token_positions)
            k = self.rope(k, token_positions)

        # 使用 tril(lower triangle) 构造下三角矩阵，下三角位置为全 1，其他位置为全 0
        mask = torch.tril(torch.ones((seq_len, seq_len), device=q.device)).bool() # 掩码矩阵 (seq_len, seq_len)：只允许看当前以及历史 token，不能看未来 token

        output_head = scaled_dot_product_attention(q, k, v, mask) # (B, n_head, S, d_head) -> (B, n_head, S, d_head)，B 和 n_head 均为批次维度，执行广播操作

        output_head = rearrange(output_head, "... num_heads seq d_head -> ... seq (num_heads d_head)") # 多头结果拼接: (B, n_heads, S, d_head) → (B, S, n_heads*d_head=d_model)
        
        output = self.w_output(output_head) # 输出层投影: (B, S, d_model) -> (B, S, d_model)
        
        return output

In [14]:
# 测试 MultiHeadSelfAttention 因果多头自注意力
batch_size = 2
seq_len = 5
d_model = 128
num_heads = 8
max_seq_len = 128
rope_theta = 10000.0

mha = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads, max_seq_len=max_seq_len, rope_theta=rope_theta)
x_in = torch.randn(batch_size, seq_len, d_model)
mha_out = mha(x_in)
print(f"MHA 输入形状： {x_in.shape}, MHA 输出形状： {mha_out.shape}")

MHA 输入形状： torch.Size([2, 5, 128]), MHA 输出形状： torch.Size([2, 5, 128])
